# BERT 中文情感分类教学实验

本 Notebook 面向本科教学，目标让学生看清楚 NLP 模型的完整流程：

```
原始文本 → 数字化表示 → TF-IDF / Tokenizer → BERT Embedding → Attention → 分类预测
```

本实验包含两条路线：

1. **传统 NLP**：TF-IDF + Logistic Regression；
2. **深度学习 NLP**：BERT + Attention + Fine-tuning。

通过这两条路线对比，可以理解为什么 BERT 不只是统计词频，还能够利用上下文和注意力机制进行语义建模。

## 你将学到什么

| 知识点 | 对应章节 |
|--------|---------|
| NLP 数据预处理与探索 | 1~3 |
| 传统文本分类（TF-IDF + LR） | 4~7 |
| BERT Tokenizer 与特殊标记（[CLS]、[SEP]、[PAD]） | 8~9 |
| BERT Embedding（文本 → 向量） | 10 |
| **Attention 可视化与热力图解读** | **11~12** |
| 模型微调（Fine-tuning） | 13~16 |
| 测试评估与迁移测试 | 17~终 |

**重点章节**：11~12 节（Attention 可视化）是本次实验最核心的内容，请仔细理解 [CLS] 标记在注意力机制中的角色。


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from huggingface_hub import hf_hub_download
from datasets import Dataset


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from transformers import (
    BertTokenizer,
    BertModel,
    BertForSequenceClassification,
    AutoTokenizer,
    AutoModelForCausalLM
)

from torch.optim import AdamW

from tqdm import tqdm

from pathlib import Path
import pandas as pd

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("当前设备:", device)
print("PyTorch版本:", torch.__version__)

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

## 1. 下载并项目化管理数据集

这一单元负责把中文情感分类数据集下载到当前项目目录，而不是放在系统缓存中。

这样做有两个教学上的好处：

- 学生可以直接在 `data/chnsenticorp/` 文件夹中看到训练集、验证集和测试集；
- 后续 Notebook、模型文件和数据文件都在同一个项目目录中，便于复现实验。

由于当前 `datasets` 新版本不再支持旧式数据集脚本，这里直接下载 Hugging Face 仓库中的 `.arrow` 文件，再转成 CSV。

In [ ]:
print("=" * 80)
print("中文情感数据集下载与项目化管理")
print("=" * 80)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "chnsenticorp"
DATA_DIR.mkdir(exist_ok=True, parents=True)

print("当前工作目录:")
print(PROJECT_DIR)

print("\n数据保存目录:")
print(DATA_DIR)

repo_id = "seamew/ChnSentiCorp"

files = {
    "train": "chn_senti_corp-train.arrow",
    "validation": "chn_senti_corp-validation.arrow",
    "test": "chn_senti_corp-test.arrow"
}

local_arrow_paths = {}

for split_name, filename in files.items():
    print(f"\n正在下载 {split_name}: {filename}")
    
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset",
        local_dir=DATA_DIR,
        local_dir_use_symlinks=False
    )
    
    local_arrow_paths[split_name] = local_path
    print("保存路径:", local_path)

# =========================
# 读取 arrow 文件
# =========================

train_dataset = Dataset.from_file(local_arrow_paths["train"])
validation_dataset = Dataset.from_file(local_arrow_paths["validation"])
test_dataset = Dataset.from_file(local_arrow_paths["test"])

train_df = train_dataset.to_pandas()
validation_df = validation_dataset.to_pandas()
test_df = test_dataset.to_pandas()

# =========================
# 保存 CSV，方便学生查看
# =========================

train_csv = DATA_DIR / "chnsenticorp_train.csv"
valid_csv = DATA_DIR / "chnsenticorp_validation.csv"
test_csv = DATA_DIR / "chnsenticorp_test.csv"

train_df.to_csv(train_csv, index=False, encoding="utf-8-sig")
validation_df.to_csv(valid_csv, index=False, encoding="utf-8-sig")
test_df.to_csv(test_csv, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("数据集处理完成")
print("=" * 80)

print("训练集数量:", len(train_df))
print("验证集数量:", len(validation_df))
print("测试集数量:", len(test_df))

print("\nCSV 文件已保存:")
print(train_csv)
print(valid_csv)
print(test_csv)

print("\n训练集前5行:")
display(train_df.head())

## 2. 读取本地 CSV 并检查数据结构

NLP 任务的输入通常是一列文本，输出是一列标签。

在这个数据集中：

- `text`：用户评论文本；
- `label=0`：消极评论；
- `label=1`：积极评论。

这一步的重点是让学生明确：**NLP 里的训练数据本质上也是一个监督学习表格**，只是输入不是图像或数值，而是自然语言文本。

In [ ]:
train_df = pd.read_csv(DATA_DIR / "chnsenticorp_train.csv")
validation_df = pd.read_csv(DATA_DIR / "chnsenticorp_validation.csv")
test_df = pd.read_csv(DATA_DIR / "chnsenticorp_test.csv")

label_names = {
    0: "Negative",
    1: "Positive"
}

train_df["label_name"] = train_df["label"].map(label_names)
validation_df["label_name"] = validation_df["label"].map(label_names)
test_df["label_name"] = test_df["label"].map(label_names)

print("=" * 80)
print("本地中文情感数据集")
print("=" * 80)

print("训练集 shape:", train_df.shape)
print("验证集 shape:", validation_df.shape)
print("测试集 shape:", test_df.shape)

print("\n训练集前6行:")
display(train_df.head(6))

print("\n训练集标签分布:")
display(train_df["label_name"].value_counts())

## 3. 文本长度统计

BERT 不能直接处理任意长度的文本。训练前必须指定最大长度 `max_length`。

- 文本太长：会被截断；
- 文本太短：会用 `[PAD]` 补齐。

因此，观察文本长度分布是 NLP 数据预处理中的重要步骤。本实验后面统一使用 `max_length=64`，这是本科教学中比较合适的折中设置。

In [ ]:
# =========================
# 文本长度统计
# =========================

train_df["text_length"] = train_df["text"].apply(len)
validation_df["text_length"] = validation_df["text"].apply(len)
test_df["text_length"] = test_df["text"].apply(len)

print("=" * 80)
print("文本长度统计")
print("=" * 80)

display(train_df["text_length"].describe())

plt.figure(figsize=(7, 5))
plt.hist(train_df["text_length"], bins=40)
plt.xlabel("Text Length")
plt.ylabel("Count")
plt.title("训练集文本长度分布")
plt.grid(True)
plt.show()

print("\n课堂解释:")
print("文本长度会影响 BERT 输入长度。")
print("后续我们会设置 max_length=64。")
print("过长文本会被截断，较短文本会被 padding 补齐。")

## 4. 传统 NLP：TF-IDF 特征构建

TF-IDF 是传统文本分类中非常经典的方法。它的核心思想是：

```text
一个词在当前文本中经常出现，但在所有文本中不常见，说明它可能更重要。
```

这里将每条评论转换为一个 5000 维的稀疏向量。

这一步可以帮助学生理解：即使是文本，进入机器学习模型前也必须被转换为数值特征。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

train_small = train_df.sample(n=min(3000, len(train_df)), random_state=42).reset_index(drop=True)
test_small = test_df.sample(n=min(800, len(test_df)), random_state=42).reset_index(drop=True)

print("=" * 80)
print("传统方法：TF-IDF 特征构建")
print("=" * 80)

print("训练样本数:", len(train_small))
print("测试样本数:", len(test_small))

vectorizer = TfidfVectorizer(
    max_features=5000,
    token_pattern=r"(?u)\b\w+\b"
)

X_train_tfidf = vectorizer.fit_transform(train_small["text"])
X_test_tfidf = vectorizer.transform(test_small["text"])

y_train_base = train_small["label"].values
y_test_base = test_small["label"].values

print("\nTF-IDF 训练矩阵 shape:", X_train_tfidf.shape)
print("TF-IDF 测试矩阵 shape:", X_test_tfidf.shape)

print("\n解释:")
print("每一行是一条评论文本。")
print("每一列是一个文本特征。")
print("TF-IDF 数值越大，说明该词对当前文本越重要。")

## 5. 查看 TF-IDF 表格

这一单元将前 6 条评论的 TF-IDF 特征打印成表格。

教学重点：

- 每一行是一条评论；
- 每一列是一个词或字符特征；
- 表格中的数值表示该特征的重要程度。

这和前面图像任务中的“图像展平成像素表格”类似，本质都是把复杂对象转成机器可处理的数值矩阵。

In [ ]:
feature_names = vectorizer.get_feature_names_out()

tfidf_preview = pd.DataFrame(
    X_train_tfidf[:6, :30].toarray(),
    columns=feature_names[:30]
)

tfidf_preview["label"] = y_train_base[:6]
tfidf_preview["label_name"] = [label_names[i] for i in y_train_base[:6]]

print("=" * 80)
print("传统 NLP：TF-IDF 表格数据")
print("=" * 80)

print("DataFrame shape:")
print(tfidf_preview.shape)

display(tfidf_preview)

print("\n课堂解释:")
print("传统 NLP 会把文本转换成一个数值表格。")
print("这种方法简单有效，但难以理解上下文语义。")

## 6. 传统文本分类器：Logistic Regression

这一单元使用 Logistic Regression 完成情感分类。

它代表传统 NLP 流程：

```text
文本 → TF-IDF 特征 → 线性分类器 → 积极/消极
```

优点是简单、快速、可解释。

局限是它主要依赖词频统计，难以理解上下文、转折关系和词义变化。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("=" * 80)
print("传统方法：TF-IDF + Logistic Regression")
print("=" * 80)

baseline_model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    verbose=1
)

print("开始训练 Logistic Regression 文本分类器...")

baseline_model.fit(
    X_train_tfidf,
    y_train_base
)

print("训练完成。")

y_pred_base = baseline_model.predict(
    X_test_tfidf
)

baseline_acc = accuracy_score(
    y_test_base,
    y_pred_base
)

print("\n传统方法准确率:")
print(baseline_acc)

print("\n分类报告:")
print(classification_report(
    y_test_base,
    y_pred_base,
    target_names=["Negative", "Positive"]
))

## 7. 混淆矩阵：观察传统模型错在哪里

准确率只能告诉我们整体正确率，混淆矩阵可以进一步显示：

- 多少消极评论被正确预测为消极；
- 多少积极评论被正确预测为积极；
- 哪些样本发生了混淆。

这一步延续了前面 CIFAR-10 图像分类实验中的评估思路。

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test_base,
    y_pred_base
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

disp.plot(cmap="Blues")

plt.title("TF-IDF + Logistic Regression 混淆矩阵")

plt.show()

print("\n课堂解释:")
print("对角线表示预测正确。")
print("非对角线表示分类错误。")

## 8. BERT Tokenizer：文本如何进入 BERT

BERT 不能直接读取中文句子。它需要先经过 Tokenizer。

Tokenizer 会完成三件事：

```
原始文本 → token → input_ids → attention_mask
```

其中：

- `input_ids` 是每个 token 在词表中的编号；
- `attention_mask` 用来告诉模型哪些位置是真实文本，哪些位置是 padding。

这一单元是理解 BERT 输入格式的关键。

---

### 认识 BERT 的特殊标记（Special Tokens）

在下方的输出结果中，你会看到几个特殊的标记。它们是 BERT 处理文本的基础构件，请务必理解它们各自的作用：

#### [CLS]（Classification Token — 分类标记）
- **位置**：每个输入序列的**第一个 token**（词表编号 ID = 101）
- **作用**：用来**聚合整个句子的语义信息**。BERT 将 [CLS] 对应位置的最后一层隐藏向量作为"句向量"，送入分类器进行情感判断
- **教学比喻**：可以把 [CLS] 想象成一个"情报收集员"。它一开始是空白的，在通过 12 层 Transformer 的过程中，通过注意力机制从句子中的每个词那里"收集情报"，最终形成一个能代表整句话含义的向量
- **为什么叫 CLS**：因为它是用来做 **CL**a**S**sification（分类）的

#### [SEP]（Separator Token — 分隔标记）
- **位置**：**每个句子的末尾**（ID = 102）
- **作用**：**标记"句子到这里结束"**。对于情感分类这种单句任务，[SEP] 告诉模型句子边界在哪；对于句子对任务（如问答、文本蕴含），两句之间也需要用 [SEP] 隔开
- **教学理解**：就像我们写作文时的句号，告诉读者（模型）一句话在哪里结束

#### [PAD]（Padding Token — 填充标记）
- **位置**：序列末尾的填充区域（ID = 0）
- **作用**：**补齐长度**。BERT 要求一个 batch 内的所有句子长度相同，短句子末尾自动补 [PAD] 到统一的 `max_length`
- **关键机制**：`attention_mask` 中，真实 token 标记为 1，[PAD] 标记为 0。这样模型在计算注意力时就会**忽略这些占位符**，不会让无意义的填充影响判断

#### [UNK]（Unknown Token — 未知标记，选读）
- 当遇到词表中没有的词时（如英文缩写、罕见字），Tokenizer 会把它替换为 [UNK]

---

### 解码后的完整格式

当你看到解码结果：

```
[CLS] 这 家 餐 厅 环 境 很 好 ， 但 是 服 务 很 差 。 [SEP] [PAD] [PAD] ...
```

请按以下方式理解：

| 位置 | token | 含义 |
|------|-------|------|
| 0 | [CLS] | 分类聚合标记，将用于情感预测 |
| 1~16 | 这 家 餐 厅 ... 差 。 | 实际的中文字 token（中文 BERT 基于单字切分） |
| 17 | [SEP] | 句子结束标记 |
| 18~31 | [PAD] ... | 填充占位符，不参与注意力计算 |

接下来，观察输出的 `attention_mask` 也可以验证：前 18 个位置是 1（有效 token），后面全是 0（填充）。


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

print("当前工作目录:")
print(PROJECT_DIR)

print("\n模型目录:")
print(MODEL_DIR)

# =========================
# BERT Tokenizer 演示
# 下载到当前项目目录
# =========================

from transformers import BertTokenizer

tokenizer_path = MODEL_DIR / "bert-base-chinese"

# 第一次运行会下载
if not tokenizer_path.exists():

    print("首次下载 BERT Tokenizer...")

    tokenizer = BertTokenizer.from_pretrained(
        "bert-base-chinese"
    )

    tokenizer.save_pretrained(
        tokenizer_path
    )

    print("Tokenizer 已保存到:")
    print(tokenizer_path)

else:

    print("检测到本地 Tokenizer")

    tokenizer = BertTokenizer.from_pretrained(
        tokenizer_path
    )

sample_text = "这家餐厅环境很好，但是服务很差。"

tokens = tokenizer.tokenize(sample_text)

encoding = tokenizer(
    sample_text,
    max_length=32,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

print("=" * 80)
print("BERT Tokenizer 演示")
print("=" * 80)

print("原始文本:")
print(sample_text)

print("\nToken结果:")
print(tokens)

print("\ninput_ids:")
print(encoding["input_ids"])

print("\nattention_mask:")
print(encoding["attention_mask"])

print("\n解码回文本:")
print(tokenizer.decode(encoding["input_ids"][0]))

## 9. 下载并加载 BERT Base Model

这一单元将 `bert-base-chinese` 模型保存到当前项目目录的 `models/` 文件夹中。

这样做可以避免模型散落在 Hugging Face 默认缓存目录里，便于课堂演示和项目管理。

这里加载的是 BERT 的基础编码器，用于后续查看 Embedding 和 Attention。

In [ ]:
from transformers import BertModel
from pathlib import Path

PROJECT_DIR = Path.cwd()

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

bert_model_path = MODEL_DIR / "bert-base-chinese-model"

# 第一次运行下载
if not bert_model_path.exists():

    print("首次下载 BERT Base Model...")

    bert_base = BertModel.from_pretrained(
        "bert-base-chinese"
    )

    bert_base.save_pretrained(
        bert_model_path
    )

    print("模型已保存:")
    print(bert_model_path)

else:

    print("检测到本地模型")

    bert_base = BertModel.from_pretrained(
        bert_model_path
    )

bert_base = bert_base.to(device)

print("\n当前设备:")
print(device)

## 10. BERT Embedding：token 如何变成向量

Tokenizer 只能把文本变成 token id，但 token id 本身只是编号（比如 6821 代表"这"），这些编号之间没有语义上的连续关系——"餐厅"和"服务"的编号并不相近。

BERT Embedding 层负责把每个 token 的编号转换成一个 **768 维的稠密向量**：

```
token id → 768 维语义向量
```

### 教学重点：理解张量形状的变化

| 数据 | 形状 | 解释 |
|------|------|------|
| `input_ids` | [1, 32] | 1 句话，32 个 token 编号 |
| `embedding_output` | [1, 32, 768] | 1 句话，32 个 token，每个 token 用 768 维向量表示 |

也就是说：**每个 token 都从"一个数字"变成了"一个 768 维的向量"**。

### [CLS] 对应的嵌入向量

注意第一个 token（**位置 0**）对应的就是 **[CLS] 的嵌入向量**：

- **当前阶段**：Embedding 层只是"查表映射"，此时每个 token 的向量还是**孤立的**，不包含上下文信息
- **后续阶段**：经过 12 层 Transformer 编码后，[CLS] 位置的向量会通过注意力机制聚合所有 token 的语义，形成"句向量"

可以这样理解整个流程：

```
原始文本 
    ↓
Tokenizer → input_ids（每个字变成编号）
    ↓
Embedding → 每个编号变成 768 维向量（查表，不含上下文）
    ↓
12 层 Transformer（通过 Attention 融入上下文信息）
    ↓
[CLS] 位置的向量 → 分类器 → 情感预测
```

**你现在就在见证 NLP 的核心转变：离散的文本符号，被转换成了计算机可以处理的连续数值向量。**


In [ ]:
sample_text = "这家餐厅环境很好，但是服务很差。"

encoding = tokenizer(
    sample_text,
    max_length=32,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

bert_base.eval()

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

with torch.no_grad():
    embedding_output = bert_base.embeddings(input_ids)

print("=" * 80)
print("BERT Embedding 输出")
print("=" * 80)

print("原始文本:")
print(sample_text)

print("\ninput_ids shape:")
print(input_ids.shape)

print("\nattention_mask shape:")
print(attention_mask.shape)

print("\nembedding_output shape:")
print(embedding_output.shape)

print("\n第1个 token 的前10维向量:")
print(embedding_output[0, 0, :10].detach().cpu().numpy())

print("\n解释:")
print("[batch_size, sequence_length, hidden_size]")
print("hidden_size = 768")
print("即每个 token 被表示为一个 768 维向量。")

## 11. Attention 可视化：BERT 如何"关注"上下文

这是本实验**最重要的教学环节之一**。

Attention 机制是 BERT 的核心能力。Attention 矩阵中的每个元素表示：

```
当前 token 对另一个 token 的关注程度 / 权重
```

### 为什么要可视化 Attention？

对于句子：

```
这家餐厅环境很好，但是服务很差。
```

人类很容易看出这里有转折关系："环境很好"但"服务很差"。但模型是如何"看"出来的？

通过 Attention 热力图，我们可以直接"看到"模型内部的工作方式——模型在判断情感时，**哪些词更受关注、哪些词之间建立了联系**。

### [CLS] 在 Attention 中的关键角色

观察即将生成的热力图，请特别关注 **第一行（[CLS] 行）** 和 **第一列（[CLS] 列）**：

- **第一行（[CLS] 行）**：表示 [CLS] 这个 token 对句子中每个其他 token 的关注程度
  - 如果 [CLS] 对"很好"的关注权重高，说明模型认为"很好"是判断情感的关键信息
  - 如果 [CLS] 对"但是"关注权重高，说明模型捕捉到了转折关系
  - **这正是 [CLS] 收集句子语义的过程**——通过高权重关注重要词汇来聚合信息

- **第一列（[CLS] 列）**：表示句子中其他 token 对 [CLS] 的关注程度
  - 通常权重较低，因为普通词汇不需要关注这个特殊标记

### 解读热力图

在热力图中：

| 维度 | 含义 |
|------|------|
| **横轴（X轴）** | 被关注的 token（也叫 Key——相当于"被查询的内容"） |
| **纵轴（Y轴）** | 当前"主动关注"的 token（也叫 Query——相当于"发出查询的提问者"） |
| **颜色深浅** | 注意力权重大小，颜色越深（值越大），关注程度越高 |

### 在这句话中值得观察的重点

1. **"但是"**：转折词通常会获得较高关注，因为它标志着情感的变化
2. **"环境" ↔ "很好"**：主语和褒义词之间的关联
3. **"服务" ↔ "很差"**：主语和贬义词之间的关联
4. **[CLS] 行**：观察 [CLS] 更关注哪些词（情感词、转折词）


In [ ]:
bert_attn = BertModel.from_pretrained(
    bert_model_path,
    output_attentions=True
).to(device)

bert_attn.eval()

text = "这家餐厅环境很好，但是服务很差。"

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=32,
    padding="max_length",
    truncation=True
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = bert_attn(**inputs)

attentions = outputs.attentions

print("=" * 80)
print("BERT Attention 信息")
print("=" * 80)

print("Attention 层数:")
print(len(attentions))

print("\n第1层 attention shape:")
print(attentions[0].shape)

print("\nshape 含义:")
print("[batch_size, head数量, seq_len, seq_len]")

# 取出有效 token，不显示 padding 部分
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0].cpu()
)

valid_len = int(
    inputs["attention_mask"][0].sum().item()
)

tokens_valid = tokens[:valid_len]

# 取第1层、第1个head
attn_matrix = attentions[0][0, 0, :valid_len, :valid_len].cpu().numpy()

plt.figure(figsize=(9, 8))
plt.imshow(attn_matrix)
plt.xticks(range(valid_len), tokens_valid, rotation=90)
plt.yticks(range(valid_len), tokens_valid)
plt.colorbar()
plt.title("BERT Attention Heatmap\nLayer 1 - Head 1")
plt.tight_layout()
plt.show()

print("\n课堂解释:")
print("横轴表示被关注的 token。")
print("纵轴表示当前 token。")
print("颜色越深，表示注意力权重越高。")

## 12. 多头注意力：不同 Head 关注不同关系

BERT 的每一层不是只计算一个 Attention，而是使用 **多个 Attention Head**（bert-base-chinese 每层有 12 个 Head）。

### 为什么需要"多头"？

可以把它理解为：**模型从多个不同的角度观察同一句话**。

就像一组评委同时评价一道菜：

| 评委 | 关注点 |
|------|--------|
| 评委 A | 味道怎么样 |
| 评委 B | 价格贵不贵 |
| 评委 C | 服务好不好 |
| 评委 D | 环境怎么样 |

每个评委（Head）从自己的角度给出判断，最后综合所有评委的意见形成最终结论。

### 不同 Head 可能关注的语言关系

在 BERT 中，不同的 Attention Head 在学习过程中会自动"分工"，各自捕捉不同类型的语言模式：

- **Head A**：关注情感词（如"很好"、"很差" → 抓住评价核心）
- **Head B**：关注修饰关系（如"非常"→"好"，"太"→"差" → 理解程度）
- **Head C**：关注转折结构（如"但是"前后的对比 → 理解情感变化）
- **Head D**：关注句法结构（主语←→谓语关系）

### 对比查看不同 Head

下面的代码绘制了**第 1 层（Layer 1）** 的前 4 个 Head 的热力图。

请对比观察它们之间的差异：

- **有些 Head** 的注意力分布比较均匀（倾向于关注所有 token → 捕捉全局信息）
- **有些 Head** 集中在少数关键的词上（倾向关注少数 token → 抽取关键信息）
- **有些 Head** 可能特别关注 [CLS] 与情感词之间的连接（服务于分类任务）

本科教学中不需要把每个 Head 解释得过度绝对，但核心结论是：

> **多头注意力使 BERT 能够同时从多个角度捕捉语言关系，这是它比传统 NLP 方法理解能力更强的重要原因之一。**


In [ ]:
layer_idx = 0
num_heads = 4

plt.figure(figsize=(14, 12))

for head_idx in range(num_heads):

    attn_matrix = attentions[layer_idx][0, head_idx, :valid_len, :valid_len].cpu().numpy()

    plt.subplot(2, 2, head_idx + 1)

    plt.imshow(attn_matrix)

    plt.xticks(
        range(valid_len),
        tokens_valid,
        rotation=90
    )

    plt.yticks(
        range(valid_len),
        tokens_valid
    )

    plt.title(
        f"Layer {layer_idx + 1}, Head {head_idx + 1}"
    )

plt.tight_layout()

plt.show()

print("\n课堂解释:")
print("BERT 使用多头注意力。")
print("不同 Head 可以从不同角度关注句子中的词。")
print("例如情感词、转折词、主语、修饰关系等。")

## 13. 加载 BERT 分类模型

前面的 BERT Base Model 只负责产生文本表示（即把每个 token 变成语义向量）。

情感分类还需要在 BERT 后面接一个**分类头（Classification Head）**：

```
[CLS] 向量（768维） → Linear（768→2） → [Negative得分, Positive得分]
```

这里使用 `BertForSequenceClassification`，它已经在 BERT 编码器的基础上内置了这个分类头。

### 关键概念：为什么只取 [CLS] 向量？

你可能会问：为什么只取 [CLS] 而不取所有 token 的平均？

这是因为：

1. **[CLS] 是专门设计的聚合标记**：BERT 在预训练时，[CLS] 就承担了聚合句子级信息的任务
2. **注意力机制保证了信息汇聚**：经过 12 层 Transformer，[CLS] 通过注意力"看到"了所有其他 token
3. **效率更高**：只用一个向量比用所有向量更高效，而且效果更好

### 模型结构一览

下面打印的是完整的模型结构。你不需要记住所有细节，但要能看出三个主要部分：

| 组件 | 参数规模 | 作用 |
|------|---------|------|
| **BERT Embeddings** | ~23M | 把 token 编号映射为向量 |
| **BERT Encoder（12层）** | ~85M | 通过多头注意力编码上下文语义 |
| **Classifier** | ~1.5K | 把 [CLS] 向量映射到 2 个类别 |

注意：此时分类头的参数是**随机初始化的**（输出中会看到 `MISSING` 警告），因此模型还不能做情感分类——需要先进行微调（fine-tuning）。


In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    bert_model_path,
    num_labels=2
).to(device)

print(model)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n模型总参数量:", total_params)
print("可训练参数量:", trainable_params)

print("\n课堂解释:")
print("BERT 主体负责理解文本语义。")
print("最后的 classifier 线性层负责输出 Negative / Positive 两类得分。")

## 14. BERT 前向传播检查

这一单元只取少量样本，检查 BERT 分类模型的一次完整前向传播。

教学重点：

- `input_ids`：文本编号矩阵；
- `attention_mask`：有效 token 标记（1=有效，0=填充）；
- `logits`：模型输出的两个类别的原始得分；
- `loss`：当前预测与真实标签之间的差异。

这一步和前面 CNN 实验中的"输入 batch → 输出 logits"是同一个思想。

---

### BERT 情感分类的内部流程

当输入经过 BERT 的全部 12 层 Transformer 编码后，分类过程如下：

```
输入文本
    ↓
Tokenizer → [CLS] 这 家 餐 ... [SEP] [PAD] ...
    ↓
12 层 Transformer 编码（每层都包含多头注意力）
    ↓
取出 [CLS] 位置的最终向量（形状: [768]）
    ↓
分类器：Linear(768 → 2)线性层
    ↓
输出 logits（形状: [2]） → [Negative得分, Positive得分]
    ↓
Softmax → 概率 → 取最大值对应的类别
```

### 为什么 [CLS] 如此重要？

- [CLS] 是**连接 BERT 编码器和分类头的桥梁**
- 分类器的输入只来自 **[CLS] 位置**，而不是所有 token 的平均
- 这意味着 BERT **必须通过注意力机制**把整句话的信息压缩到 [CLS] 这一个位置上
- 这正是你之前在 Attention 热力图中看到 [CLS] 行关注各个词的原因

### 理解 logits

输出的 logits 形状为 `[batch_size, num_classes]`，这里是 `[8, 2]`：

- `logits[0]` = Negative 类别的得分
- `logits[1]` = Positive 类别的得分
- 得分越高的类别，就是模型的预测结果

由于模型还没有经过微调（fine-tuning），两个类别的得分会比较接近（都在 0.25 左右），说明模型此时还无法准确判断情感。


In [ ]:
from torch.utils.data import Dataset, DataLoader

# 取少量样本演示
demo_df = train_df.head(16).copy()

class DemoDataset(Dataset):

    def __init__(self, dataframe, tokenizer):

        self.df = dataframe
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = self.df.iloc[idx]["text"]

        label = int(
            self.df.iloc[idx]["label"]
        )

        encoding = tokenizer(
            text,
            max_length=64,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "label":
                torch.tensor(label)
        }

demo_dataset = DemoDataset(
    demo_df,
    tokenizer
)

demo_loader = DataLoader(
    demo_dataset,
    batch_size=8,
    shuffle=False
)

batch = next(iter(demo_loader))

input_ids = batch["input_ids"].to(device)

attention_mask = batch["attention_mask"].to(device)

labels = batch["label"].to(device)

# =========================
# 前向传播
# =========================

model.eval()

with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

loss = outputs.loss

logits = outputs.logits

preds = torch.argmax(
    logits,
    dim=1
)

print("=" * 80)
print("BERT 前向传播检查")
print("=" * 80)

print("input_ids shape:")
print(input_ids.shape)

print("\nattention_mask shape:")
print(attention_mask.shape)

print("\nlogits shape:")
print(logits.shape)

print("\nloss:")
print(loss.item())

print("\n前5个真实标签:")
print(labels[:5].cpu().numpy())

print("\n前5个预测标签:")
print(preds[:5].cpu().numpy())

print("\n第一条样本 logits:")
print(logits[0].cpu().numpy())

print("\n课堂解释:")
print("logits 的形状为 [batch_size, num_classes]")
print("这里是 [8,2]")
print("对应 Negative 和 Positive 两个类别")

## 15. 构建 BERT Dataset 和 DataLoader

深度学习训练不能一次把所有文本都送入模型，而是按 batch（小批量）训练。

这一单元完成数据管线的搭建：

```
DataFrame → Dataset → DataLoader
```

每个样本包含三个部分：

| 字段 | 形状 | 解释 |
|------|------|------|
| `input_ids` | [max_length] | 每个 token 的词表编号 |
| `attention_mask` | [max_length] | 有效 token 标记（1=有效，0=填充） |
| `label` | 标量 | 情感类别（0=Negative, 1=Positive） |

---

### input_ids 中的特殊标记

以 `max_length=64` 为例，每个样本的 input_ids 序列结构如下：

```
位置 0:     [CLS]（ID=101）    ← 分类聚合标记，最终用于情感预测
位置 1~N:   实际句子 token      ← "这"、"家"、"餐"、"厅"...
位置 N+1:   [SEP]（ID=102）    ← 句子结束标记
位置 N+2~63: [PAD]（ID=0）     ← 填充占位符
```

### attention_mask 的作用

attention_mask 与 input_ids 一一对应：

```
input_ids:     [101, 6821, 2157, ..., 102, 0, 0, 0, ...]
attention_mask: [ 1,    1,    1, ...,   1, 0, 0, 0, ...]
```

- **真实 token**（包括 [CLS]、句中词语、[SEP]）→ attention_mask = **1**
- **填充 token**（[PAD]）→ attention_mask = **0**

这样，模型在计算 Self-Attention 时，就会把 attention_mask=0 的位置"屏蔽"掉，确保无意义的填充词不会影响模型的理解。


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SentimentDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=64
    ):

        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        text = str(row["text"])

        label = int(row["label"])

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }


# 为了 MacBook 教学演示速度，只取部分样本
bert_train_df = train_df.sample(
    n=min(2000, len(train_df)),
    random_state=42
).reset_index(drop=True)

bert_test_df = test_df.sample(
    n=min(500, len(test_df)),
    random_state=42
).reset_index(drop=True)

train_dataset = SentimentDataset(
    bert_train_df,
    tokenizer,
    max_length=64
)

test_dataset = SentimentDataset(
    bert_test_df,
    tokenizer,
    max_length=64
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("=" * 80)
print("BERT DataLoader 构建完成")
print("=" * 80)

print("训练样本数量:", len(train_dataset))
print("测试样本数量:", len(test_dataset))
print("训练 batch 数量:", len(train_loader))
print("测试 batch 数量:", len(test_loader))

batch = next(iter(train_loader))

print("\n一个 batch 的 input_ids shape:")
print(batch["input_ids"].shape)

print("\n一个 batch 的 attention_mask shape:")
print(batch["attention_mask"].shape)

print("\n一个 batch 的 label shape:")
print(batch["label"].shape)

print("\n前1条 input_ids:")
print(batch["input_ids"][0])

print("\n前1条 attention_mask:")
print(batch["attention_mask"][0])

print("\n前1条 label:")
print(batch["label"][0])

print("\n课堂解释:")
print("input_ids 是 token 编号。")
print("attention_mask 用于区分真实 token 和 padding。")
print("label 是情感类别，0=Negative，1=Positive。")

## 16. 微调 BERT 完成情感分类

这一单元开始**真正训练模型**。

### 什么是 Fine-tuning（微调）？

BERT 已经在大规模中文语料（维基百科、新闻等）上完成了**预训练（Pre-training）**，学会了通用的语言理解能力。

现在，我们只需要在它基础上做少量的"针对性训练"，让它适应**情感分类**这个具体任务。

这就好比你请了一位博学的语言学家（预训练 BERT），只需要教他两天"如何判断评论情感"（微调），他就能很好地完成这个任务——而不是从教他认字开始。

### 微调过程中的关键机制

在微调过程中，**整个模型**的参数都会更新，包括：

1. **BERT 编码器**：调整自身参数，使 [CLS] 更精准地关注情感相关词汇
2. **分类头（Classifier）**：从随机初始化开始，学习如何根据 [CLS] 向量判断情感

微调完成后，[CLS] 的向量会变成这样：

```
微调前：[CLS] 向量中，"不错"和"不好"的区分度不够
微调后：[CLS] 向量会"更关注"情感词，"不错"和"不好"产生完全不同的表示
```

### 实验配置

| 参数 | 值 | 说明 |
|------|-----|------|
| 训练样本 | 2000 条 | 为教学演示速度适当缩减 |
| 测试样本 | 500 条 | 用于评估模型效果 |
| max_length | 64 | 每条文本统一长度 |
| batch_size | 8 | 每次输入 8 条文本 |
| 学习率 | 2e-5 | 微调通常使用很小的学习率 |
| Early Stopping | 2 轮 | 测试准确率连续 2 轮不提升则停止 |


In [ ]:
from torch.optim import AdamW
import copy
import os

print("=" * 80)
print("BERT 中文情感分类训练")
print("=" * 80)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

num_epochs = 5
patience = 2

best_test_acc = 0.0
epochs_no_improve = 0
best_model_state = None

train_losses = []
train_accuracies = []
test_accuracies = []


def evaluate_bert(model, data_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for batch in data_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    return correct / total


for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{num_epochs}"
    )

    for batch in loop:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        logits = outputs.logits

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        preds = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        loop.set_postfix(
            loss=loss.item()
        )

    epoch_loss = running_loss / total

    train_acc = correct / total

    test_acc = evaluate_bert(
        model,
        test_loader
    )

    train_losses.append(epoch_loss)

    train_accuracies.append(train_acc)

    test_accuracies.append(test_acc)

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")
    print(f"Train Loss: {epoch_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Test Acc  : {test_acc:.4f}")

    if test_acc > best_test_acc:

        best_test_acc = test_acc

        epochs_no_improve = 0

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        print("测试准确率提升，保存当前最佳模型。")

    else:

        epochs_no_improve += 1

        print(f"测试准确率未提升，累计 {epochs_no_improve} 次。")

    print("-" * 50)

    if epochs_no_improve >= patience:

        print("触发 Early Stopping，训练提前结束。")

        break


if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

    print("已恢复最佳模型。")


save_dir = "./saved_models_bert"

os.makedirs(
    save_dir,
    exist_ok=True
)

model_save_path = os.path.join(
    save_dir,
    "bert_chinese_sentiment.pth"
)

tokenizer_save_dir = os.path.join(
    save_dir,
    "bert_tokenizer"
)

checkpoint = {

    "model_state_dict": model.state_dict(),

    "model_name": "bert-base-chinese",

    "task_type": "text_classification",

    "label_names": label_names,

    "best_test_accuracy": best_test_acc,

    "train_losses": train_losses,

    "train_accuracies": train_accuracies,

    "test_accuracies": test_accuracies
}

torch.save(
    checkpoint,
    model_save_path
)

tokenizer.save_pretrained(
    tokenizer_save_dir
)

print("\n模型已保存:")
print(model_save_path)

print("\nTokenizer 已保存:")
print(tokenizer_save_dir)

print("\n最佳测试准确率:")
print(best_test_acc)

In [ ]:
actual_epochs = len(train_losses)

history_df = pd.DataFrame({
    "epoch": range(1, actual_epochs + 1),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "test_accuracy": test_accuracies
})

print("=" * 80)
print("BERT 训练历史")
print("=" * 80)

display(history_df)

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BERT训练损失变化")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(history_df["epoch"], history_df["test_accuracy"], marker="o", label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("BERT准确率变化")
plt.legend()
plt.grid(True)
plt.show()

## 测试集整体评估

模型训练完成后，我们需要在测试集上进行全面评估。

### 评估指标说明

| 指标 | 含义 |
|------|------|
| **Precision（精确率）** | 模型预测为 Positive 的样本中，真正是 Positive 的比例 |
| **Recall（召回率）** | 真正的 Positive 样本中，模型成功找出来的比例 |
| **F1-score** | Precision 和 Recall 的调和平均，综合评价指标 |

### 概率输出

除了预测类别，模型还会输出每个类别的**概率值**：

- `negative_prob`：属于消极类的概率（0~1）
- `positive_prob`：属于积极类的概率（0~1）

如果 `positive_prob = 0.92`，说明模型对"这条评论是正面的"这个判断非常自信。

通过查看概率值，我们可以发现模型的"犹豫"样本——比如 `positive_prob = 0.52` 说明模型也不太确定。


In [ ]:
model.eval()

all_preds = []
all_labels = []
all_probs = []
all_texts = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# 注意：bert_test_df 顺序与 test_loader 一致，因为 test_loader shuffle=False
all_texts = bert_test_df["text"].tolist()

bert_acc = accuracy_score(all_labels, all_preds)

print("=" * 80)
print("BERT 测试集整体结果")
print("=" * 80)

print("BERT Test Accuracy:", bert_acc)

print("\n分类报告:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=["Negative", "Positive"]
))

bert_result_df = pd.DataFrame({
    "text": all_texts,
    "true_label": all_labels,
    "true_sentiment": [label_names[i] for i in all_labels],
    "pred_label": all_preds,
    "pred_sentiment": [label_names[i] for i in all_preds],
    "negative_prob": [p[0] for p in all_probs],
    "positive_prob": [p[1] for p in all_probs],
})

display(bert_result_df.head(10))

## BERT 混淆矩阵

混淆矩阵（Confusion Matrix）是分类模型评估中最直观的工具之一。

### 如何阅读混淆矩阵

```
             预测 Negative    预测 Positive
真实 Negative      TN               FP
真实 Positive      FN               TP
```

- **TN（True Negative）**：正确识别为消极 → 对角线 ✅
- **TP（True Positive）**：正确识别为积极 → 对角线 ✅
- **FP（False Positive，误报）**：实际消极，模型误判为积极 → 非对角线 ❌
- **FN（False Negative，漏报）**：实际积极，模型误判为消极 → 非对角线 ❌

通过混淆矩阵，我们可以分析模型的"偏好"——是否倾向于把某些类型的评论判错。


In [ ]:
cm = confusion_matrix(
    all_labels,
    all_preds
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

disp.plot(cmap="Blues")

plt.title("BERT 中文情感分类混淆矩阵")

plt.show()

print("\n课堂解释:")
print("对角线表示预测正确。")
print("非对角线表示预测错误。")
print("如果 Negative 被预测成 Positive，说明模型把负面评论误判为正面。")
print("如果 Positive 被预测成 Negative，说明模型把正面评论误判为负面。")

## 方法对比：传统 NLP vs BERT

现在我们把两种方法放在一起对比，从多个维度分析它们的差异。

### 为什么 BERT 表现更好？

下面我们用贯穿本实验的知识来解释：

| 维度 | TF-IDF + LR | BERT Fine-tuning |
|------|-------------|------------------|
| **输入形式** | 稀疏词频向量（5000维，大部分为0） | 稠密语义向量（[CLS] 最终表示，768维） |
| **上下文理解** | ❌ 只看词频，忽略顺序和上下文 | ✅ 通过 **Attention 机制** 理解每个词在上下文中的含义 |
| **转折处理** | ❌ "不错" 和 "不好吃" 同时出现 → 混淆 | ✅ [CLS] 通过注意力权重区分前后情感，"但是" 标记了转折 |
| **泛化能力** | 弱，依赖训练集词汇 | 强，可迁移到未见过的新领域 |
| **可解释性** | 可以查看哪些词权重高 | 可以通过 **Attention 热力图** 看到模型关注什么 |

### 核心结论

TF-IDF 是**统计词的重要性**，BERT 是**理解词在上下文中的含义**。

这正是 Attention 机制带来的革命性变化——模型不再是"看词数词"，而是"理解句子"。


In [ ]:
compare_df = pd.DataFrame({
    "方法": [
        "TF-IDF + Logistic Regression",
        "BERT Fine-tuning"
    ],
    "输入形式": [
        "稀疏词频特征",
        "Token ID + Attention Mask"
    ],
    "是否利用上下文": [
        "弱",
        "强"
    ],
    "测试准确率": [
        baseline_acc,
        bert_acc
    ]
})

print("=" * 80)
print("传统方法 vs BERT")
print("=" * 80)

display(compare_df)

plt.figure(figsize=(7, 5))
plt.bar(
    compare_df["方法"],
    compare_df["测试准确率"]
)

plt.ylabel("Accuracy")
plt.title("传统文本分类器 vs BERT")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.grid(True)
plt.show()

print("\n课堂解释:")
print("TF-IDF主要统计词出现的重要性。")
print("BERT不仅看词，还会结合上下文和注意力机制。")
print("因此 BERT 更适合处理转折、否定、语义依赖等复杂文本。")

In [ ]:
# =========================
# 人工输入文本预测
# =========================

def predict_sentiment(text):

    model.eval()

    encoding = tokenizer(
        text,
        max_length=64,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)

    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )[0]

        pred = torch.argmax(
            probs
        ).item()

    return {
        "text": text,
        "prediction": label_names[pred],
        "negative_prob": probs[0].item(),
        "positive_prob": probs[1].item()
    }

custom_texts = [
    "这家餐厅非常好吃，下次还会再来。",
    "服务态度太差了，再也不会来了。",
    "房间很干净，位置也不错。",
    "价格太贵，体验一般。",
    "虽然房间有点小，但是服务非常好。",
    "环境不错，但是菜真的不好吃。"
]

custom_results = [
    predict_sentiment(text)
    for text in custom_texts
]

custom_result_df = pd.DataFrame(custom_results)

display(custom_result_df)

## 新场景文本测试
### 部署

In [ ]:
import torch
from transformers import (
    BertForSequenceClassification,
    BertTokenizer
)

from pathlib import Path

MODEL_FILE = Path(
    "/Users/linzuhong/Downloads/HUTBCourse_AI_Undergraduate-main/8_nlp/saved_models_bert/bert_chinese_sentiment.pth"
)

TOKENIZER_DIR = Path(
    "/Users/linzuhong/Downloads/HUTBCourse_AI_Undergraduate-main/8_nlp/saved_models_bert/bert_tokenizer"
)

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("加载已训练BERT模型")
print("=" * 80)

checkpoint = torch.load(
    MODEL_FILE,
    map_location=device
)

label_names = checkpoint["label_names"]

model = BertForSequenceClassification.from_pretrained(
    "bert-base-chinese",
    num_labels=2
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)

model.eval()

tokenizer = BertTokenizer.from_pretrained(
    TOKENIZER_DIR
)

print("模型加载完成")
print("最佳测试准确率:",
      checkpoint["best_test_accuracy"])

### 自定义预测函数

In [ ]:
# =========================
# 情感预测函数
# =========================

def predict_sentiment(text):

    encoding = tokenizer(
        text,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)

    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )[0]

        pred = torch.argmax(
            probs
        ).item()

    return {
        "text": text,
        "prediction": label_names[pred],
        "negative_prob": probs[0].item(),
        "positive_prob": probs[1].item()
    }

### 测试

In [ ]:
new_texts = [

    "这部电影拍得非常精彩，值得推荐。",

    "剧情太拖沓了，我看到一半就不想看了。",

    "老师讲课非常清楚，收获很大。",

    "课程内容太浅，没有学到什么东西。",

    "这个手机续航很强，运行也流畅。",

    "电池发热严重，体验很差。",

    "这个大模型回答问题非常专业。",

    "生成的结果漏洞百出。",

    "环境不错，但是菜真的不好吃。",

    "虽然价格有点贵，但是整体体验很好。"
]

results = [
    predict_sentiment(text)
    for text in new_texts
]

result_df = pd.DataFrame(results)

display(result_df)

## 模型迁移测试（Domain Transfer Test）

训练集来源：

- 酒店评论

模型学习目标：

- 判断评论是正面还是负面

---

## 问题

如果模型只见过酒店评论：

```text
房间很干净
服务很好
早餐丰富
```

那么它能否判断：

```text
这部电影拍得非常精彩

这个手机续航很差

老师讲课非常清楚
```

这些从未出现过的新领域文本？

---

## 两种可能

### 情况1：模型只是记忆训练集

那么：

```text
电影
手机
课程
```

这些词没有见过，

模型将无法判断。

---

### 情况2：模型学习到了语言规律

例如：

```text
精彩
推荐
满意
喜欢
```

通常表达积极情绪。

```text
糟糕
失望
差劲
讨厌
```

通常表达消极情绪。

那么模型就能迁移到新的领域。

---

## 本实验目标

验证：

BERT是否真正学到了语言规律，
而不仅仅是在记忆酒店评论。

## 迁移测试数据

In [ ]:
transfer_texts = [

    ("这部电影拍得非常精彩，值得推荐。", "Positive"),

    ("剧情太拖沓了，我看到一半就不想看了。", "Negative"),

    ("老师讲课非常清楚，收获很大。", "Positive"),

    ("课程内容太浅，没有学到什么东西。", "Negative"),

    ("这个手机续航很强，运行很流畅。", "Positive"),

    ("电池发热严重，体验很差。", "Negative"),

    ("这个大模型回答问题非常专业。", "Positive"),

    ("生成结果漏洞百出。", "Negative")
]

transfer_df = pd.DataFrame(
    transfer_texts,
    columns=["text","expected"]
)

display(transfer_df)

## 模型迁移测试预测

In [ ]:
results = []

for text, expected in transfer_texts:

    pred = predict_sentiment(text)

    results.append({
        "text": text,
        "expected": expected,
        "prediction": pred["prediction"],
        "negative_prob": pred["negative_prob"],
        "positive_prob": pred["positive_prob"]
    })

transfer_result_df = pd.DataFrame(results)

display(transfer_result_df)

correct = (
    transfer_result_df["expected"]
    ==
    transfer_result_df["prediction"]
).sum()

acc = correct / len(transfer_result_df)

print("=" * 80)
print("迁移测试结果")
print("=" * 80)

print("Accuracy:", acc)

## Attention 与转折句分析

下面这句话非常经典：

```
环境不错，但是菜真的不好吃。
```

### 人类会怎么判断？

人类读这句话，会判断为**负面评价**。原因很清楚："但是"之后的评价比"但是"之前的更重要——"不错"是好的，但"不好吃"才是最终结论。

### 传统方法为什么容易犯错？

传统 TF-IDF 方法看到的是：

```
环境中："不错" → 正面词汇 ✓
"好吃" → 正面词汇 ✓
```

于是模型可能误判为**正面**。它无法理解"但是"二字带来的语义转折。

### BERT 如何正确理解？

BERT 通过 **Attention 机制**动态决定哪些词应该被重点关注：

1. **[CLS] token** 会关注情感关键词——通过注意力"看到""不好吃"的权重可能会高于"不错"
2. **"但是"** 作为转折词，在 Attention 中会连接前后两个分句，帮助模型理解情感变化
3. **"不好吃"** 作为最终评价的核心，会获得较高的注意力权重

推理过程大致如下：

```
[CLS] → 关注 → "不错" ✓（前半句正面）
[CLS] → 关注 → "但是" ✓（有转折！）
[CLS] → 关注 → "不好吃" ✓（最终落脚点是负面）
最终判断：Negative ← "但是"之后的语义权重更高
```

### Attention 热力图验证

接下来，我们使用前面加载的 BERT 模型（`bert_attn`），为这句话绘制 Attention 热力图。

请特别关注：

- **[CLS] 行（第一行）**：[CLS] 对哪些词的关注权重最高？
- **"但是" 行**：它的注意力是如何连接前后两个分句的？
- **"不错" 和 "不好吃"**：哪个在 [CLS] 行中获得更高的关注权重？

这正是 **Attention 机制的核心价值——让模型学会"该看哪里"**。


## 转折句 Attention 分析 — 代码执行

以下代码将上一单元介绍的转折句 "环境不错，但是菜真的不好吃。" 输入 BERT，并提取其注意力权重绘制热力图。

### 代码说明

这段代码使用了之前加载的 `bert_attn` 模型（`output_attentions=True`），主要步骤为：

1. **Tokenizer 编码**：将句子转为 `input_ids` 和 `attention_mask`
2. **前向传播**：通过 BERT 模型，获取所有层的注意力权重
3. **提取注意力矩阵**：取 **最后一层（Layer 11）**、**第 1 个 Head** 的注意力
4. **裁剪有效区域**：只显示有效 token（去掉 [PAD] 部分）
5. **绘制热力图**：横纵轴都标注对应的 token

### 观察要点

运行下面的代码后，请仔细观察热力图：

- **[CLS] 行**：BERT 的"情报收集员"重点关注了哪些词？
- **"但是" 行**：它的注意力是否连接了"环境不错"和"菜真的不好吃"？
- **对角线**：每个 token 是否倾向于关注自己？（Self-Attention 中常见的模式）
- **颜色深浅**：哪对 (token_i, token_j) 之间的注意力最高？

理解这些之后，你就真正掌握了 **BERT Attention 机制的核心思想**。


In [ ]:
sentence = "环境不错，但是菜真的不好吃。"

inputs = tokenizer(
    sentence,
    return_tensors="pt",
    max_length=32,
    truncation=True,
    padding="max_length"
)

inputs = {
    k:v.to(device)
    for k,v in inputs.items()
}

with torch.no_grad():

    outputs = bert_attn(**inputs)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

valid_len = int(
    inputs["attention_mask"][0]
    .sum()
)

tokens = tokens[:valid_len]

attention = outputs.attentions[-1][0,0]

attention = attention[
    :valid_len,
    :valid_len
].cpu().numpy()

plt.figure(figsize=(10,8))

plt.imshow(attention)

plt.xticks(
    range(valid_len),
    tokens,
    rotation=90
)

plt.yticks(
    range(valid_len),
    tokens
)

plt.colorbar()

plt.title(
    "Attention Heatmap\n环境不错，但是菜真的不好吃"
)

plt.tight_layout()

plt.show()

## 18. 增加 GPT 架构：从“理解文本”到“生成文本”

前面的 BERT 部分主要用于完成中文情感分类：

```text
输入完整句子
↓
BERT 双向编码
↓
[CLS] 表示整句话
↓
分类头
↓
Positive / Negative
```

BERT 的核心优势是**理解文本**，它可以同时看见一个词左边和右边的上下文，因此适合：

- 文本分类；
- 情感分析；
- 句子匹配；
- 文本理解类任务。

但是，BERT 不是典型的文本生成模型。它的结构是 Encoder，不是 Decoder，不能像 GPT 那样自然地执行：

```text
给定前文
↓
预测下一个 token
↓
把预测结果接回输入
↓
继续预测下一个 token
```

这正是 GPT 的自回归生成机制。

这里的“自回归”不是前面房价预测中的数值回归，而是：

> 模型每一步都利用自己已经生成的内容，继续预测下一个 token。

因此，本节加入 GPT 架构，用于说明：

```text
BERT 更擅长理解；
GPT 更擅长生成。
```

## 18.1 GPT 架构详解：从输入到生成的完整流程

前面的 BERT 采用了 Transformer **Encoder（编码器）**，它能够同时看到一句话中所有词的**双向上下文**。

GPT 则采用了不同的架构——**Transformer Decoder（解码器）**，它只能**从左到右**逐个读词。

---

### GPT Decoder 的逐层处理流程

```text
输入文本 → Token Embedding + Position Embedding
    ↓
┌──────────────────────────────────────────┐
│  Decoder Block × N 层                    │
│  ┌─────────────────────────────────┐    │
│  │  ① Masked Multi-Head            │    │
│  │     Self-Attention              │    │
│  │     (因果注意力 / Causal Attn)   │    │ ← 只能用前文
│  └─────────────────────────────────┘    │
│            ↓                            │
│  ┌─────────────────────────────────┐    │
│  │  ② LayerNorm + 残差连接         │    │ ← Add & Norm
│  └─────────────────────────────────┘    │
│            ↓                            │
│  ┌─────────────────────────────────┐    │
│  │  ③ Feed-Forward Neural          │    │
│  │     Network（两层 MLP）          │    │ ← 非线性变换
│  └─────────────────────────────────┘    │
│            ↓                            │
│  ┌─────────────────────────────────┐    │
│  │  ④ LayerNorm + 残差连接         │    │ ← Add & Norm
│  └─────────────────────────────────┘    │
└──────────────────────────────────────────┘
    ↓
LM Head（语言模型头，即 Linear + Softmax）
    ↓
词表上的概率分布 → 从中选择下一个 token
```

---

### 核心区别：Causal Attention（因果注意力 / 掩码注意力）

GPT 与 BERT **最根本的架构差异**在于 Attention 的可见范围：

| 特性 | BERT (Encoder) | GPT (Decoder) |
|------|---------------|---------------|
| **Attention 方向** | 双向（可以看前后文） | 单向（只能看前文） |
| **注意力矩阵形状** | 全部位置互相可见 | **下三角矩阵**（上三角被掩码为 0） |
| **数学公式** | Softmax(QKᵀ / √d) | Softmax(QKᵀ / √d + **M**) |
| **M 矩阵** | 全为 0 | 上三角为 **-∞**（掩码遮罩） |
| **生成能力** | ❌ 不适合逐词生成 | ✅ 天然支持自回归生成 |

> 这个 Mask（掩码）确保了——模型在预测第 i 个 token 时，只能看到第 1 到第 i-1 个 token，看不到 i 及之后的 token。

#### 为什么需要因果注意力？

如果 GPT 也像 BERT 一样可以看到未来的词，那预测下一个词就变成了作弊：

```text
句子：服务很[好]  → 如果模型已经看到了[好]，
                    那预测[服务很___]时就等于提前知道了答案
```

因果注意力保证了每个位置的预测**只依赖前文**，这正是自回归的数学基础。

---

### GPT 的组件 vs BERT 的组件

| 组件 | BERT | GPT |
|------|------|-----|
| 词嵌入（Token Embedding） | ✅ | ✅ |
| 位置编码（Position Embedding） | ✅ 可学习的绝对位置编码 | ✅ 可学习的绝对位置编码 |
| 段编码（Segment Embedding） | ✅ | ❌ 不需要 |
| [CLS] / [SEP] 特殊标记 | ✅ 用于分类和分句 | ❌ 不需要 |
| Self-Attention 类型 | 双向全连接 | 因果掩码（单向） |
| 每层子层数 | 2 个（Attention + FFN） | 2 个（Masked Attention + FFN） |
| 输出头 | 分类头 / 句向量 | LM Head（映射回词表维度） |

GPT 的架构比 BERT 更简洁——它不需要为分类任务设计的特殊标记，只需要**前文 → 后文**的自回归预测能力。

---

### BERT vs GPT：一句话总结架构差异

```text
BERT：Encoder + 双向 Attention → 理解整句话 → 适合分类
GPT：  Decoder + 因果 Attention → 逐词生成 → 适合续写
```


## 19. 下载并项目化管理 GPT 模型

为了和前面的 BERT 项目保持一致，GPT 模型也保存到当前项目目录：

```text
8_nlp/
│
├── data/
├── models/
│   ├── bert-base-chinese/
│   ├── bert-base-chinese-model/
│   └── gpt2-chinese-cluecorpussmall/
└── saved_models_bert/
```

这样做有两个好处：

1. 学生可以明确看到模型文件保存在哪里；
2. 后续课堂演示不依赖 Hugging Face 默认缓存目录。

本实验使用较小的中文 GPT-2 模型：

```text
uer/gpt2-chinese-cluecorpussmall
```

它主要用于教学演示，不追求生成质量。

In [ ]:
# =========================
# 下载并加载中文 GPT 架构模型
# 模型：uer/gpt2-chinese-cluecorpussmall
# 保存到当前项目目录
# =========================

from pathlib import Path
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# 项目路径
# =========================

PROJECT_DIR = Path.cwd()

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

# =========================
# 中文 GPT 模型
# 说明：
# 这是中文通用 GPT2 模型（基于 CLUE 语料训练）。
# 体积较小（约 95M 参数），适合本科教学中演示：
# 1. GPT Decoder-only 架构
# 2. 自回归生成（Autoregressive Generation）
# 3. Next Token Prediction（下一个词预测）
# 4. Causal / Masked Self-Attention（因果注意力）
# =========================

gpt_model_name = "uer/gpt2-chinese-cluecorpussmall"

gpt_model_dir = MODEL_DIR / "gpt2-chinese-cluecorpussmall"

print("=" * 80)
print("中文 GPT 模型下载与本地管理")
print("=" * 80)

print("当前工作目录:")
print(PROJECT_DIR)

print("\n模型目录:")
print(gpt_model_dir)

# =========================
# 下载模型
# =========================

if (
    not gpt_model_dir.exists()
    or
    not any(gpt_model_dir.iterdir())
):

    print("\n首次下载中文 GPT 模型")
    print("模型较小（约 95M 参数），通常较快完成")

    snapshot_download(
        repo_id=gpt_model_name,
        local_dir=gpt_model_dir
    )

    print("\n下载完成")

else:

    print("\n检测到本地模型，直接加载")

# =========================
# Tokenizer
# =========================

gpt_tokenizer = AutoTokenizer.from_pretrained(
    gpt_model_dir
)

# GPT 类模型通常没有 pad_token
# 这里用 eos_token 作为 pad_token
if gpt_tokenizer.pad_token is None:

    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

# =========================
# GPT 模型
# =========================

gpt_lm = AutoModelForCausalLM.from_pretrained(
    gpt_model_dir
)

# =========================
# 设备
# =========================

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

gpt_lm = gpt_lm.to(device)

gpt_lm.eval()

print("\n当前设备:")
print(device)

# =========================
# 模型统计
# =========================

total_params = sum(
    p.numel()
    for p in gpt_lm.parameters()
)

print("\n模型参数量:")
print(f"{total_params:,}")

print("\n约 {:.4f} M 参数".format(
    total_params / 1e6
))

print("\n词表大小:")
print(gpt_lm.config.vocab_size)

print("\nTransformer 层数（num_hidden_layers）:")
print(gpt_lm.config.num_hidden_layers)

print("\n注意力头数（num_attention_heads）:")
print(gpt_lm.config.num_attention_heads)

print("\n隐藏层维度（hidden_size）:")
print(gpt_lm.config.hidden_size)

print("\n模型加载完成")

print("\n课堂提醒:")
print("这个模型是中文通用 GPT2（基于 CLUE 语料训练）。")
print("它适合展示 GPT 自回归生成机制、Decoder-only 架构和因果注意力。")
print("生成内容仅展示语言规律，不具备真正语义理解能力。")

In [ ]:
# =========================
# 查看 GPT 模型架构
# 展示 Decoder-only 结构
# =========================

print("=" * 80)
print("GPT Decoder-only 模型架构")
print("=" * 80)

print(gpt_lm)

print("\n" + "=" * 80)
print("GPT 模型结构分析")
print("=" * 80)

print(f"\n1. 模型类型: GPT2LMHeadModel（Decoder-only）")
print(f"2. Transformer 层数: {gpt_lm.config.num_hidden_layers}")
print(f"3. 注意力头数/层: {gpt_lm.config.num_attention_heads}")
print(f"4. 隐藏层维度: {gpt_lm.config.hidden_size}")
print(f"5. 词表大小: {gpt_lm.config.vocab_size}")
print(f"6. 最大序列长度: {gpt_lm.config.max_position_embeddings}")
print(f"7. 激活函数: {gpt_lm.config.activation_function}")

print("\n" + "=" * 80)
print("与 BERT 的架构对比")
print("=" * 80)

print(f"\n{'─'*40}")
print("BERT（本实验）: BertForSequenceClassification")
print(f"{'─'*40}")
print(f"  - 架构类型: Encoder-only")
print(f"  - Attention: 双向（全连接）")
print(f"  - 编码器层数: 12")
print(f"  - 注意力头数: 12")
print(f"  - 隐藏层维度: 768")
print(f"  - 输出: [CLS] 向量 → 分类头 → 2 分类")
print(f"  - 典型用途: 理解、分类、匹配")

print(f"\n{'─'*40}")
print("GPT（本实验）: GPT2LMHeadModel")
print(f"{'─'*40}")
print(f"  - 架构类型: Decoder-only")
print(f"  - Attention: 单向（因果掩码）")
print(f"  - 解码器层数: {gpt_lm.config.num_hidden_layers}")
print(f"  - 注意力头数: {gpt_lm.config.num_attention_heads}")
print(f"  - 隐藏层维度: {gpt_lm.config.hidden_size}")
print(f"  - 输出: LM Head → 词表({gpt_lm.config.vocab_size})上的概率分布")
print(f"  - 典型用途: 生成、续写、对话")

print("\n课堂解释:")
print("BERT 的输出是一个向量（[CLS]），用于分类。")
print("GPT 的输出是词表上的概率分布，告诉你每个词是下一个词的可能性。")
print("这就是为什么 BERT 用于理解，GPT 用于生成。")
print("\n关键问题:")
print("Q: 为什么 GPT 只用 Decoder 而不用 Encoder?")
print("A: 因为生成任务要求逐个词产生，如果用了双向 Encoder，")
print("   模型在生成第 i 个词时就能看到第 i+1 个词，变成作弊。")
print("   Decoder + Causal Attention 恰好保证了只看前文的约束。")


In [ ]:
# =========================
# GPT-2 中文文本生成演示
# =========================

prompts = [
    "这家餐馆",
    "这个人说话",
    "今天的天气"
]

for prompt in prompts:

    print("=" * 80)
    print("Prompt:")
    print(prompt)

    inputs = gpt_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = gpt_lm.generate(
            inputs.input_ids,
            max_new_tokens=30,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.9,
            pad_token_id=gpt_tokenizer.pad_token_id
        )

    generated = gpt_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\n生成结果:")
    print(generated)

## 20. BERT 与 GPT 的输入方式对比

BERT 和 GPT 都需要先把文本变成 token，再变成数字编号。

但是二者的模型目标不同：

| 模型 | 架构 | 主要任务 | 是否适合生成 |
|---|---|---|---|
| BERT | Encoder | 理解整句话 | 不适合直接生成 |
| GPT | Decoder | 根据前文预测后文 | 适合生成 |

BERT 通常使用：

```text
[CLS] 句子 [SEP]
```

其中 `[CLS]` 用于表示整句话，常用于分类任务。

GPT 不需要 `[CLS]`，因为它的目标不是“给整句话分类”，而是：

```text
看到前面的 token，预测下一个 token。
```

In [ ]:
# =========================
# BERT vs GPT Tokenizer 对比
# =========================

from transformers import BertTokenizer

print("=" * 80)
print("BERT 与 GPT Tokenizer 对比")
print("=" * 80)

# 优先加载本地 BERT tokenizer，如果不存在则从 huggingface 下载
bert_tokenizer_path = MODEL_DIR / "bert-base-chinese"

if bert_tokenizer_path.exists():
    bert_tokenizer = BertTokenizer.from_pretrained(
        bert_tokenizer_path
    )
else:
    bert_tokenizer = BertTokenizer.from_pretrained(
        "bert-base-chinese"
    )

sample_text = "这家餐厅环境很好，但是服务很差。"

bert_tokens = bert_tokenizer.tokenize(
    sample_text
)

gpt_tokens = gpt_tokenizer.tokenize(
    sample_text
)

print("原始文本:")
print(sample_text)

print("\nBERT tokens:")
print(bert_tokens)

print("\nGPT tokens:")
print(gpt_tokens)

bert_encoding = bert_tokenizer(
    sample_text,
    max_length=32,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

gpt_encoding = gpt_tokenizer(
    sample_text,
    max_length=32,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

print("\nBERT input_ids:")
print(bert_encoding["input_ids"])

print("\nGPT input_ids:")
print(gpt_encoding["input_ids"])

print("\nBERT 解码:")
print(bert_tokenizer.decode(bert_encoding["input_ids"][0]))

print("\nGPT 解码:")
print(gpt_tokenizer.decode(gpt_encoding["input_ids"][0]))

print("\n课堂解释:")
print("BERT 的输入服务于文本理解和分类。")
print("GPT 的输入服务于自回归生成。")
print("BERT 通常看完整句话；GPT 按顺序从左到右生成文本。")

## 21. GPT 自回归生成：BERT 不能直接完成的任务

情感分类中，BERT 的输出是类别：

```text
Positive / Negative
```

而 GPT 的输出是新的文本：

```text
给定：这家餐厅
生成：这家餐厅的服务很好，环境也不错……
```

GPT 的生成过程是逐步完成的：

```text
第 1 步：预测下一个 token
第 2 步：把生成的 token 接到原文本后面
第 3 步：再预测下一个 token
……
```

这叫自回归生成：

```text
x1, x2, x3 → 预测 x4
x1, x2, x3, x4 → 预测 x5
x1, x2, x3, x4, x5 → 预测 x6
```

因此，本单元重点展示 GPT 能做、而 BERT 不能自然完成的事情：

> 根据前文连续生成后文。

## 21.1 GPT 的因果注意力机制可视化

前面我们通过 BERT 看到了全连接的注意力矩阵——每个 token 可以关注所有其他 token。

GPT 的注意力矩阵则不同：它是一种**下三角矩阵**，上三角部分被掩码（置为 -∞），确保模型在预测当前位置时不能看到未来的 token。

### 因果注意力矩阵示意

对于四个 token **我 爱 你 [EOS]**：

```text
           我     爱     你    [EOS]
    我    [0.7,  -inf,  -inf,  -inf]   ← 我 只能看自己
    爱    [0.3,  0.5,   -inf,  -inf]   ← 爱 可看 [我, 爱]
    你    [0.2,  0.3,   0.4,   -inf]   ← 你 可看 [我, 爱, 你]
    [EOS] [0.1,  0.2,   0.3,   0.4]   ← [EOS] 可看所有
```

上三角的 `-inf` 表示这些位置的注意力权重被强制置为 0。

### 为什么这对于教学很重要？

对比 BERT（全连接注意力）和 GPT（因果注意力），学生可以直观理解：

| 方面 | BERT | GPT |
|------|------|-----|
| 注意力视野 | 全向（前后都看） | 单向（只看左边） |
| 对句子的理解 | 同时看到环境很好和服务很差 | 先看到环境很好，还未看到服务很差 |
| 适合任务 | 情感分类、完形填空 | 文本生成、对话 |

下面通过代码提取 GPT 某一层的注意力权重并可视化，学生可以清楚看到**下三角**的形状。


In [ ]:
# =========================
# GPT 因果注意力可视化
# 展示下三角掩码矩阵
# =========================

from transformers import GPT2Model

print("=" * 80)
print("GPT 因果注意力可视化")
print("=" * 80)

# 加载 GPT 基础模型（带 attention 输出）
gpt_base = GPT2Model.from_pretrained(
    gpt_model_dir,
    output_attentions=True
).to(device)

gpt_base.eval()

sentence = "这家餐厅环境很好，但是服务"

inputs = gpt_tokenizer(
    sentence,
    return_tensors="pt",
    max_length=24,
    truncation=True,
    padding="max_length"
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = gpt_base(**inputs)

gpt_attentions = outputs.attentions

print(f"Attention 层数: {len(gpt_attentions)}")
print(f"第 1 层 Attention shape: {gpt_attentions[0].shape}")
print(f"含义: [batch, head, seq_len, seq_len]\n")

# 获取有效 token 长度
valid_len = int(inputs["attention_mask"][0].sum().item())
tokens = gpt_tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0].cpu()
)[:valid_len]

print(f"有效 token 数量: {valid_len}")
print(f"Tokens: {tokens}\n")

# =========================
# 可视化：第 1 层，第 1 个 Head
# 观察下三角形状
# =========================

attn_first = gpt_attentions[0][0, 0, :valid_len, :valid_len].cpu().numpy()

plt.figure(figsize=(9, 8))
plt.imshow(attn_first, cmap="Blues")
plt.xticks(range(valid_len), tokens, rotation=90, fontsize=11)
plt.yticks(range(valid_len), tokens, fontsize=11)
plt.colorbar()
plt.title("GPT Causal Attention\nLayer 1 - Head 1\n"
          + "上三角全黑 = 被掩码的未来 token",
          fontsize=13)
plt.tight_layout()
plt.show()

print("观察重点:")
print("\u2705 上三角区域全黑 — 表示模型看不到未来的 token")
print("\u2705 下三角区域有颜色 — 表示每个 token 只能关注自己和前面的 token")
print()
print("这和 BERT 的 Attention 完全不同！")
print("BERT 的上三角也是有颜色的（全连接），可以看到未来。")
print()

# =========================
# 对比：同一句子的 BERT Attention
# =========================

print("=" * 80)
print("对比：BERT 双向注意力（同一句子）")
print("=" * 80)

bert_inputs = tokenizer(
    sentence,
    return_tensors="pt",
    max_length=24,
    truncation=True,
    padding="max_length"
)

bert_inputs = {k: v.to(device) for k, v in bert_inputs.items()}

with torch.no_grad():
    bert_outputs = bert_attn(**bert_inputs)

bert_valid = int(bert_inputs["attention_mask"][0].sum().item())

bert_attn_matrix = bert_outputs.attentions[0][0, 0, :bert_valid, :bert_valid].cpu().numpy()

bert_tokens = tokenizer.convert_ids_to_tokens(
    bert_inputs["input_ids"][0].cpu()
)[:bert_valid]

plt.figure(figsize=(9, 8))
plt.imshow(bert_attn_matrix, cmap="Blues")
plt.xticks(range(bert_valid), bert_tokens, rotation=90, fontsize=11)
plt.yticks(range(bert_valid), bert_tokens, fontsize=11)
plt.colorbar()
plt.title("BERT Bidirectional Attention\nLayer 1 - Head 1\n"
          + "上三角也有颜色 = 全连接，可看所有位置",
          fontsize=13)
plt.tight_layout()
plt.show()

print("\n对比结论:")
print("GPT: 下三角矩阵 → 只能看左边 → 自回归生成")
print("BERT: 全连接矩阵 → 可看所有位置 → 双向理解")


## 21.2 GPT 的概率分布与解码策略

GPT 模型预测的不是下一个词是什么，而是**下一个词是每个词的可能性有多大**——即整个词表上的概率分布。

### 从 logits 到概率分布

```text
GPT Decoder 最后一层
    ↓
LM Head（线性层）
    ↓
logits: 词表大小维度（如 13000 维）的原始得分
    ↓
Softmax
    ↓
概率分布: 所有词的概率之和 = 1
    ↓
选择策略 → 确定下一个 token
```

### 常见的解码策略

| 策略 | 方法 | 效果 |
|------|------|------|
| **Greedy Decoding（贪心解码）** | 每次选概率最高的 token | 结果确定性高，但容易重复和短视 |
| **Sampling（随机采样）** | 按概率分布随机抽取 | 结果多样，但可能输出不合理的词 |
| **Top-k Sampling** | 只在概率最高的 k 个词中采样 | 平衡多样性和合理性 |
| **Top-p (Nucleus) Sampling** | 在累积概率达到 p 的词中采样 | 动态调整候选范围 |
| **Temperature Scaling** | 调节 softmax 的平滑程度 | 控制随机性大小 |

下面的代码将可视化 GPT 在每个生成步的概率分布，帮助学生理解模型不是在背诵答案，而是在概率空间中做选择。


In [ ]:
# =========================
# 可视化 GPT 概率分布
# 展示每一步的 token 选择过程
# =========================

print("=" * 80)
print("GPT 概率分布可视化")
print("=" * 80)

demo_prompt = "这家餐厅"

inputs = gpt_tokenizer(
    demo_prompt,
    return_tensors="pt"
).to(device)

print(f"Prompt: 「{demo_prompt}」")
print()

# 只展示第一步的概率分布
with torch.no_grad():
    outputs = gpt_lm(inputs.input_ids)

# 取最后一个位置的 logits
last_logits = outputs.logits[0, -1, :]

# Softmax 得到概率
probs = torch.softmax(last_logits, dim=0)

# 取 Top-15 最高概率的 token
top_probs, top_ids = torch.topk(probs, k=15)

top_tokens = [
    gpt_tokenizer.decode([tid.item()], skip_special_tokens=True)
    for tid in top_ids
]

top_probs = top_probs.cpu().numpy()

# 绘制概率分布条形图
plt.figure(figsize=(10, 5))
bars = plt.barh(
    range(len(top_tokens)),
    top_probs,
    color="steelblue"
)

# 高亮最高概率的 bar
bars[0].set_color("crimson")

plt.yticks(
    range(len(top_tokens)),
    [f"「{t}」" if t.strip() else "「(空格)」" for t in top_tokens]
)
plt.xlabel("Probability")
plt.title(
    f"GPT Top-15 候选 Token 概率\n"
    + f"Prompt: 「{demo_prompt}」",
    fontsize=13
)
plt.tight_layout()
plt.show()

print(f"最高概率候选: 「{top_tokens[0]}」({top_probs[0]:.4f})")
print(f"第1个词概率: {top_probs[0]:.4f}")
print(f"第2个词概率: {top_probs[1]:.4f}")
print(f"第15个词概率: {top_probs[14]:.4f}")
print()
print("课堂解释:")
print("红色 bar 是概率最高的候选 token（Greedy 会选择这个）。")
print("蓝色 bar 是其他候选，加起来的总概率可能超过最高概率。")
print("因此 Sampling（采样）策略有时会选择概率不高但合起来很大的那部分。")
print()
print("关键概念:")
print("GPT 每次生成都是在概率空间中做选择——不是确定性地知道下一个词。")
print("不同的解码策略（贪心/采样/top-k/top-p/温度）就是不同的选择规则。")


In [ ]:
# =========================
# GPT 自回归文本生成演示
# 带 Prompt 级进度
# =========================

print("=" * 80)
print("GPT 自回归文本生成演示")
print("=" * 80)

prompts = [
    "这家餐厅",
    "这个产品",
    "今天天气"
]

all_generation_results = []

for prompt in tqdm(
    prompts,
    desc="GPT 文本生成进度"
):

    print(f"\n{'=' * 60}")
    print(f"Prompt: 「{prompt}」")
    print(f"{'=' * 60}")

    inputs = gpt_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    # 采样生成：带随机性
    with torch.no_grad():
        sampled_outputs = gpt_lm.generate(
            inputs.input_ids,
            max_new_tokens=20,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.9,
            pad_token_id=gpt_tokenizer.pad_token_id,
            eos_token_id=gpt_tokenizer.eos_token_id
        )

    sampled_text = gpt_tokenizer.decode(
        sampled_outputs[0],
        skip_special_tokens=True
    )

    # 贪心生成：每次选择概率最高的 token
    with torch.no_grad():
        greedy_outputs = gpt_lm.generate(
            inputs.input_ids,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=gpt_tokenizer.pad_token_id,
            eos_token_id=gpt_tokenizer.eos_token_id
        )

    greedy_text = gpt_tokenizer.decode(
        greedy_outputs[0],
        skip_special_tokens=True
    )

    print("采样生成结果:")
    print(sampled_text)

    print("\n贪心解码结果:")
    print(greedy_text)

    all_generation_results.append({
        "prompt": prompt,
        "sampled_generation": sampled_text,
        "greedy_generation": greedy_text
    })

generation_df = pd.DataFrame(
    all_generation_results
)

print("\n" + "=" * 80)
print("生成结果汇总")
print("=" * 80)

display(generation_df)

print("\n课堂解释:")
print("采样生成会引入随机性，所以每次结果可能不同。")
print("贪心解码每次选择概率最高的 token，因此结果更确定。")

In [ ]:
# =========================
# 温度参数对生成的影响
# =========================

print("=" * 80)
print("温度（Temperature）参数对生成的影响")
print("=" * 80)

temp_prompt = "今天天气"

temperatures = [0.2, 0.8, 1.5]

for temp in temperatures:

    inputs = gpt_tokenizer(
        temp_prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = gpt_lm.generate(
            inputs.input_ids,
            max_new_tokens=20,
            do_sample=True,
            temperature=temp,
            top_k=40,
            top_p=0.9,
            pad_token_id=gpt_tokenizer.pad_token_id,
            eos_token_id=gpt_tokenizer.eos_token_id
        )

    generated = gpt_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print(f"\n{'─' * 50}")
    print(f"Temperature = {temp}")
    print(f"{'─' * 50}")
    print(generated)

print("\n" + "=" * 80)
print("温度参数解释")
print("=" * 80)
print()
print("T = 0.2（低温）: 概率分布更陡峭，更倾向于选最高概率词 → 更确定性")
print("   生成结果往往比较固定、安全")
print()
print("T = 0.8（常温）: 保持原始概率分布，有一定随机性")
print("   结果有变化但基本合理")
print()
print("T = 1.5（高温）: 概率分布被拉平，低概率词也有机会选到 → 更多样化")
print("   结果可能跳跃甚至不连贯")
print()
print("教学比喻:")
print("温度就像创造力调节旋钮——")
print("  低温 → 严谨保守，总是选最稳妥的词")
print("  高温 → 天马行空，可能会选意想不到的词")


## 22. 手动演示 GPT 的逐 Token 生成过程

直接调用 `generate()` 只能看到最终结果。

为了让学生真正理解“自回归”，下面不用 `generate()`，而是手动执行：

```text
输入 prompt
↓
模型预测下一个 token
↓
选出概率最高的 token
↓
把它拼接到原文本
↓
继续预测
```

这样可以清楚看到 GPT 是如何一步一步“续写”的。

In [ ]:
# =========================
# GPT 逐 Token 自回归生成过程
# 手动循环，适合本科教学
# =========================

print("=" * 80)
print("GPT 逐 Token 自回归生成过程")
print("=" * 80)

demo_prompt = "这家餐厅"

inputs = gpt_tokenizer(
    demo_prompt,
    return_tensors="pt"
).to(device)

generated_ids = inputs.input_ids

generation_steps = []

for step in tqdm(
    range(20),
    desc="逐Token生成进度"
):

    with torch.no_grad():
        outputs = gpt_lm(
            generated_ids
        )

    # 取最后一个位置的 logits
    next_token_logits = outputs.logits[:, -1, :]

    # 计算概率
    next_token_probs = torch.softmax(
        next_token_logits,
        dim=-1
    )

    # 取概率最高的 token
    next_token_id = torch.argmax(
        next_token_probs,
        dim=-1
    ).unsqueeze(0)

    # 查看 top-5 候选 token，帮助学生理解“模型是在选择”
    top_probs, top_ids = torch.topk(
        next_token_probs[0],
        k=5
    )

    top_candidates = []
    for prob, token_id in zip(top_probs, top_ids):
        token_text = gpt_tokenizer.decode(
            [int(token_id)],
            skip_special_tokens=True
        )
        top_candidates.append(
            f"{token_text}({float(prob):.4f})"
        )

    # 拼接到已有序列
    generated_ids = torch.cat(
        [
            generated_ids,
            next_token_id
        ],
        dim=1
    )

    next_token_text = gpt_tokenizer.decode(
        next_token_id[0],
        skip_special_tokens=True
    )

    current_text = gpt_tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    generation_steps.append({
        "step": step + 1,
        "new_token_id": int(next_token_id.item()),
        "new_token": next_token_text,
        "top5_candidates": " | ".join(top_candidates),
        "current_text": current_text
    })

steps_df = pd.DataFrame(
    generation_steps
)

display(steps_df)

print("\n最终生成文本:")
print(steps_df.iloc[-1]["current_text"])

print("\n课堂解释:")
print("每一行代表 GPT 生成的一个步骤。")
print("new_token 是当前步骤真正接到文本后面的 token。")
print("top5_candidates 表示模型当时认为最可能出现的5个候选 token。")
print("current_text 表示把新 token 拼接后得到的当前文本。")

## 23. BERT 与 GPT 架构对比总结

### 1. 架构核心差异

| 对比项 | BERT | GPT |
|---|---|---|
| **全称** | Bidirectional Encoder Representations from Transformers | Generative Pre-trained Transformer |
| **架构** | Transformer **Encoder** | Transformer **Decoder** |
| **注意力类型** | 双向全连接 Attention | 因果掩码（Causal）Attention |
| **信息方向** | 可以同时看左右上下文 | 只能从左到右看前文 |
| **注意力矩阵形状** | 全连接（所有位置互相可见） | 下三角（上三角被掩码） |
| **典型任务** | 分类、理解、匹配、NER | 生成、续写、对话、翻译 |
| **输入方式** | `[CLS] + 句子 + [SEP]` | 前文 prompt（无需特殊标记） |
| **输出形式** | [CLS] 向量或类别 logits | 词表上的概率分布 |
| **是否适合生成** | ❌ 不适合直接生成 | ✅ 天然支持自回归生成 |
| **预训练目标** | Masked Language Model (MLM) + Next Sentence Prediction (NSP) | Causal Language Model (CLM) — 预测下一个 token |

### 2. 本实验中的对比

| 维度 | BERT 实验 | GPT 实验 |
|---|---|---|
| **模型** | `bert-base-chinese` | `gpt2-chinese-cluecorpussmall` |
| **参数量** | ~102M | ~95M |
| **Transformer 层数** | 12 层 Encoder | 12 层 Decoder |
| **隐藏层维度** | 768 | 768 |
| **注意力头数** | 12 | 12 |
| **本实验任务** | 情感分类（Positive / Negative） | 自回归文本生成（续写） |
| **输出形状** | [batch, 2]（两类得分） | [batch, seq_len, vocab_size]（词表概率） |
| **训练方式** | Fine-tuning（有监督微调） | Pre-trained + 推理（直接生成） |

### 3. 一句话总结

```text
BERT 更像是在读完整句子后做题（理解）；
GPT 更像是在一个词一个词地写作文（生成）。
```

### 4. 思维导图：NLP 模型路线图

```text
传统 NLP                           深度学习 NLP
┌──────────────────┐          ┌──────────────────────────────┐
│  TF-IDF + LR     │          │         Transformer           │
│                   │          │         /          \          │
│  稀疏词频特征    │          │    Encoder        Decoder     │
│  + 线性分类器    │          │       │              │        │
│                   │          │    BERT           GPT         │
│  优: 简单快速    │          │       │              │        │
│  缺: 无视上下文  │          │  理解分类       文本生成      │
└──────────────────┘          └──────────────────────────────┘
```

### 5. 思考题

1. 如果让 BERT 做文本生成，会遇到什么问题？
2. 如果让 GPT 做情感分类（不经过任何改造），为什么表现可能不如 BERT？
3. 你能否想到一个任务，既需要 BERT 的理解能力，又需要 GPT 的生成能力？
4. 为什么 GPT 的注意力矩阵是下三角？如果去掉这个掩码会发生什么？
5. 温度 T=0 时，GPT 的生成结果会是什么样？和 T=2.0 有什么不同？


## 24. 现代大语言模型（LLM）：GPT 架构的规模化扩展

本实验前面的 GPT-2 模型只有 **95M 参数**、12 层 Decoder，而今天的大语言模型（如 GPT-4、Qwen、Llama）本质上仍是**同样的 Decoder-only 架构**，只是规模扩大了数千倍：

### 从 GPT-2 到 GPT-4：同样的架构，更大的规模

| 模型 | 参数规模 | Decoder 层数 | 隐藏维度 | 训练数据 |
|------|---------|-------------|---------|---------|
| GPT-2（本实验） | 95M | 12 | 768 | 通用中文语料 |
| GPT-3 / Qwen-7B | 7B | 32 | 4096 | 万亿 token 级 |
| GPT-4 / Qwen-72B | 数百 B | 80+ | ~8192 | 多模态、多语言 |

> 1B = 10 亿参数，95M 的 GPT-2 与 7B 的 GPT-3 之间差了 **70 多倍**的参数。

### 架构上的关键改进

虽然核心架构仍是 Decoder-only，但现代 LLM 引入了若干重要改进：

| 改进 | 作用 |
|------|------|
| **Rotary Position Embedding (RoPE)** | 更好的相对位置编码，支持更长的上下文 |
| **Grouped Query Attention (GQA)** | 减少 KV 缓存，加速推理 |
| **SwiGLU / GeGLU 激活函数** | 替代 ReLU，提升 FFN 层表达能力 |
| **RMS LayerNorm** | 替代标准 LayerNorm，更稳定、更高效 |
| **Instruction Tuning（指令微调）** | 在对话数据上微调，让模型学会遵循指令 |
| **RLHF（人类反馈强化学习）** | 让模型输出更符合人类偏好 |

### 能力涌现：规模带来的质变

当模型规模超过某个临界点（约 10B 参数），LLM 会展现出小模型不具备的能力：

- **In-context Learning（上下文学习）**：给几个例子就能做新任务，无需微调
- **Chain-of-Thought（思维链）**：能写出推理步骤来解决复杂问题
- **Instruction Following（指令遵循）**：理解自然语言指令并执行

下面的代码将连接本地部署的 Qwen3.5:4b 模型（4B 参数），演示现代 LLM 在对话生成和分类场景中的应用。


In [ ]:
# =========================
# 现代 LLM：多轮对话演示
# 连接本地 Ollama API
# 展示 GPT Decoder-only 架构在对话场景中的应用
# =========================

import requests
import json

# Ollama API 地址（根据实际部署修改）
OLLAMA_URL = "http://10.161.138.150:11434/api/generate"
MODEL_NAME = "qwen3.5:4b"

print("=" * 80)
print("现代 LLM 多轮对话演示")
print("=" * 80)
print(f"模型: {MODEL_NAME}")
print(f"架构: Decoder-only Transformer（和本实验的 GPT-2 相同架构）")
print(f"规模: 约 40 亿参数（对比 GPT-2 的 0.95 亿参数）")
print("=" * 80)

# 先测试连接
print("\n正在测试连接...")
try:
    test_resp = requests.post(
        OLLAMA_URL,
        json={"model": MODEL_NAME, "prompt": "Hello", "stream": False},
        timeout=10
    )
    print(f"✅ 连接成功！状态码: {test_resp.status_code}")
except Exception as e:
    print(f"❌ 连接失败: {e}")
    print("请确认 Ollama 服务已启动，或修改 OLLAMA_URL 地址。")
    print("\n课堂解释: 此处演示的是远程 LLM API 调用方式。")
    print("现代 LLM 通常以 API 服务的形式部署（如 OpenAI API、本地 Ollama 等）。")
    exit()

def chat_with_llm():
    """多轮对话：上下文通过 history 累积传递"""
    print("\n" + "=" * 80)
    print("进入多轮对话模式（输入 exit 退出）")
    print("=" * 80)
    print("尝试输入: 解释自回归生成、对比 BERT 和 GPT、情感分类等\n")

    history = ""  # 保存对话上下文

    while True:
        user_input = input("\n你: ")

        if user_input.lower() == "exit":
            print("退出对话")
            break

        # 拼接上下文（关键！）
        # 这正是 GPT 自回归过程的教学体现：
        # 每次把历史对话作为前文，让模型预测下一个 token
        history += f"用户: {user_input}\n助手:"

        payload = {
            "model": MODEL_NAME,
            "prompt": history,
            "stream": False,
            "options": {
                "temperature": 0.7,
                "top_k": 40,
                "top_p": 0.9
            }
        }

        try:
            resp = requests.post(
                OLLAMA_URL,
                json=payload,
                timeout=120
            )
            resp.raise_for_status()

            data = resp.json()
            answer = data.get("response", "").strip()

            print(f"\n{'─' * 50}")
            print(f"🤖 {MODEL_NAME}:")
            print(f"{'─' * 50}")
            print(answer)

            # 把回答加入上下文，实现多轮对话
            history += f"{answer}\n"

        except Exception as e:
            print(f"❌ 请求失败: {e}")

print("\n课堂解释:")
print("多轮对话的本质就是 GPT 的自回归生成：")
print("  Step 1: 把当前对话历史作为 prompt")
print("  Step 2: 模型逐 token 生成回答")
print("  Step 3: 将回答拼接到历史中")
print("  Step 4: 用户输入新问题 → 回到 Step 1")
print()
print("这和 Cell 22 中 GPT 逐 Token 生成的原理完全一致！")
print("区别仅在于此处有用户的交互式输入。")

chat_with_llm()

## 25. 使用 LLM 做文本分类：In-context Learning

本实验前面使用 BERT 做情感分类的方法是 **Fine-tuning（微调）**——在预训练模型基础上添加分类头，用标注数据更新参数。

现代 LLM 提供了另一种方式：**In-context Learning（上下文学习）**——不更新模型参数，而是通过在 prompt 中提供指令和示例，让模型直接输出分类结果。

### Fine-tuning vs In-context Learning

| 对比项 | BERT Fine-tuning | LLM In-context Learning |
|------|-----------------|------------------------|
| **是否更新参数** | ✅ 更新所有参数 | ❌ 不更新参数，只设计 prompt |
| **需要标注数据** | ✅ 需要大量标注数据 | ❌ 零样本：不需要；少样本：几个示例即可 |
| **训练时间** | 数小时（GPU） | 零（只需推理） |
| **分类准确率** | 对特定任务通常更高 | 灵活但可能不如专用模型 |
| **灵活性** | 固定任务，换任务需重新训练 | 同一模型可做任意分类任务 |
| **成本** | 训练需要 GPU | 仅推理成本，但大模型单次推理成本高 |

### 三种 Prompt 策略

```text
零样本（Zero-shot）：
  Prompt = "判断以下评论的情感是正向还是负向：{文本}"

少样本（Few-shot）：
  Prompt = "正向：房间很干净\n负向：服务太差了\n正向：{文本}"

思维链（Chain-of-Thought）：
  Prompt = "分析以下评论的情感。先指出积极词和消极词，再给出结论：{文本}"
```

下面的代码将使用 Qwen3.5:4b 完成同样的情感分类任务，并与前面 BERT 微调的结果做对比。


In [ ]:
# =========================
# 使用 LLM 做情感分类
# 对比 BERT Fine-tuning vs LLM In-context Learning
# =========================

import requests

OLLAMA_URL = "http://10.161.138.150:11434/api/generate"
MODEL_NAME = "qwen3.5:4b"

print("=" * 80)
print("LLM 情感分类：In-context Learning")
print("=" * 80)

# 测试连接
try:
    requests.post(
        OLLAMA_URL,
        json={"model": MODEL_NAME, "prompt": "test", "stream": False},
        timeout=10
    )
    print("✅ 连接成功\n")
except Exception as e:
    print(f"❌ 连接失败: {e}")
    print("请确认 Ollama 服务已启动。跳过此演示。")
    exit()


def classify_with_llm(text, strategy="zero_shot"):
    """使用 LLM 进行情感分类"""

    if strategy == "zero_shot":
        prompt = f"""请判断以下中文评论的情感极性。
只输出 "Positive" 或 "Negative"，不要输出其他内容。

评论：{text}
情感："""

    elif strategy == "few_shot":
        prompt = f"""下面是一些评论及其情感分类：

评论：房间非常宽敞，住得很舒服。
情感：Positive

评论：服务态度太差了，不会再来了。
情感：Negative

评论：位置很好，交通方便。
情感：Positive

评论：价格太贵，性价比很低。
情感：Negative

现在请分类以下评论，只输出 "Positive" 或 "Negative"：

评论：{text}
情感："""

    else:  # chain_of_thought
        prompt = f"""分析以下评论的情感极性。
先找出评论中的积极词汇和消极词汇，然后给出最终判断。

评论：{text}
分析："""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.1  # 分类任务用低温度，提高确定性
        }
    }

    resp = requests.post(OLLAMA_URL, json=payload, timeout=60)
    data = resp.json()
    answer = data.get("response", "").strip()

    # 提取 Positive / Negative
    if "Positive" in answer:
        return "Positive"
    elif "Negative" in answer:
        return "Negative"
    else:
        return f"不确定({answer[:30]})"


# =========================
# 测试文本
# =========================

test_sentences = [
    ("这家餐厅非常好吃，下次还会再来。", "Positive"),
    ("服务态度太差了，再也不会来了。", "Negative"),
    ("环境不错，但是菜真的不好吃。", "Negative"),  # 转折句
    ("虽然房间有点小，但是服务非常好。", "Positive"),  # 转折句
]

print(f"{'文本':<40} {'真实':<10} {'零样本':<12} {'少样本':<12}")
print("-" * 80)

zero_correct = 0
few_correct = 0

for text, expected in test_sentences:
    zero = classify_with_llm(text, "zero_shot")
    few = classify_with_llm(text, "few_shot")

    display_text = text[:35] + "..." if len(text) > 35 else text
    print(f"{display_text:<40} {expected:<10} {zero:<12} {few:<12}")

    if zero == expected:
        zero_correct += 1
    if few == expected:
        few_correct += 1

print()
print(f"零样本准确率: {zero_correct}/{len(test_sentences)}")
print(f"少样本准确率: {few_correct}/{len(test_sentences)}")

print("\n" + "=" * 80)
print("与前面 BERT 微调结果的对比")
print("=" * 80)
print()
print("BERT Fine-tuning：")
print("  - 需要数千条标注数据进行训练")
print("  - 训练时间约 5-10 分钟（GPU）")
print("  - 测试准确率 ~90%+")
print()
print("LLM In-context Learning：")
print("  - 不需要训练数据（零样本）或仅需要几个示例（少样本）")
print("  - 无训练时间，直接推理")
print("  - 对简单任务表现良好，但复杂任务可能需要更多示例或思维链")
print()
print("教学要点:")
print("1. BERT 微调 = 小模型 + 大量数据 + 训练 → 特定任务专家")
print("2. LLM 提示 = 大模型 + 无训练 + 推理 → 通用任务通才")
print("3. 在实际应用中，两者不是替代关系而是互补关系。")


## 26. 本实验全景回顾

从 TF-IDF 到 BERT，再到 GPT，再到现代 LLM，本实验带领大家走过了 NLP 发展的一条主线：

### 技术路线图

```text
1950s                   2017                  2018                2020+         2023+
词频统计          Transformer          BERT / GPT           GPT-3         GPT-4 / Qwen
   │                   │                   │                    │               │
   ▼                   ▼                   ▼                    ▼               ▼
┌──────┐         ┌──────────┐        ┌────────┐          ┌────────┐       ┌──────────┐
│TF-IDF│   →    │ Attention│   →    │ Encoder│ 分类  →  │Decoder │  →   │  Decoder │
│ + LR │         │ Is All   │        │(BERT)  │ 微调     │(GPT-3) │       │ (LLM)    │
│      │         │ You Need │        │        │          │  + 指令 │       │  + RLHF  │
└──────┘         └──────────┘        └────────┘          └────────┘       └──────────┘
```

### 核心概念回顾

| 概念 | 本实验对应单元 | 一句话理解 |
|------|--------------|-----------|
| **Tokenization** | Cell 8, 20 | 把文本变成数字编号 |
| **Embedding** | Cell 10 | 把编号变成语义向量 |
| **Self-Attention** | Cell 11, 21 | 让每个词关联上下文中的其他词 |
| **Multi-Head Attention** | Cell 12 | 从多个角度同时关注句子 |
| **Causal Attention** | Cell 21.1 | GPT 只能看左边，不能看右边 |
| **自回归生成** | Cell 22, 24 | 一个词一个词地预测下一个 |
| **Fine-tuning** | Cell 16 | 用标注数据微调模型参数 |
| **In-context Learning** | Cell 25 | 设计 prompt，不更新参数 |
| **温度参数** | Cell 21.2 | 控制生成结果的随机性 |

### 三句话总结

1. **TF-IDF + LR**：把文本变成词频统计做分类——简单但不懂语义。
2. **BERT**：用双向 Attention 理解上下文——专注文本理解，适合分类。
3. **GPT / LLM**：用因果 Attention 从做自回归生成——一个词一个词地写，能做对话和分类。

### 展望

- **多模态 LLM**：GPT-4o / Qwen-VL 不仅能读文字，还能看图片、听声音
- **Agent（智能体）**：LLM 结合工具调用（搜索、计算、代码执行），成为主动执行任务的智能体
- **RAG（检索增强生成）**：LLM 结合知识库，既保留生成能力又获得事实准确性

> 本实验的所有核心概念——Attention、Embedding、Autoregressive Generation——都是理解这些前沿技术的基础。
